# Stage 1. Cohort board

Audience: Admin, weekly. One screen, answers *is the study working and within protocol compliance*.

Five benchmark tiles (value, Wilson 95% CI, target rule, 14-day sparkline), the
enrolment funnel, the MRT integrity strip, and the alert feed.

Two rules the layer enforces and these plots inherit: a rate is published only
above `RATE_MIN_PARTICIPANTS` (10) and `RATE_MIN_UNITS` (30) - below that the
tile shows raw counts and no percentage; and a benchmark with no data source is
marked unmeasurable, never reported as zero.

In [1]:
# Every notebook under backend/dashboard/stages/ shares one data layer: the Django
# bootstrap, the SYNTHETIC_DATA switch, the fourteen frames and every compute
# function. Set SYNTHETIC_DATA in monitor_common.py to swap fixture for ORM.
from monitor_common import *

print("data source:", describe_source())

data source: fixture fixture_cohort.json - 14 participants, seed 17, anchored 2026-09-16


In [2]:
# The whole board is one computation. Everything below reads from it.
board = cohort_board("all")
print(f'phase {board["phase"]} | as of {board["as_of"]} | {board["n_participants"]} participants')

phase all | as of 2026-09-16 | 14 participants


In [3]:
# STAGE 1A — BENCHMARK CARD (BULLET + WILSON CI + 14-DAY SPARKLINE)
# Column map: rate, numerator, denominator. The denominator is drawn, not just
# used - a rising rate during the Phase 1 ramp is otherwise indistinguishable
# from a growing cohort.
SPARK_COLUMN = {
    "slot_coverage": ("slot_coverage", "slots_covered", "slots_expected"),
    "prompt_response": ("prompt_response", "responded_n", "delivered_n"),
    "wear": ("wear_pass_rate", "wear_days_met", "wear_days_scored"),
}

# A benchmark is met / not met only when the interval says so. Colouring on the
# point estimate alone contradicts the CI drawn beside it: 73% with a CI of
# 68-78 against a 75% target is undecided, not failed, and at n=5 almost
# everything is undecided. Red on week 2 makes a PI change recruitment on noise.
BENCHMARK_BADGE = {"met": "good", "not met": "critical", "undecided": "warning",
                   "suppressed": "warning", "unmeasurable": "warning"}
BENCHMARK_DETAIL = {
    "met": "CI clears target",
    "not met": "CI below target",
    "undecided": "CI spans target - not yet decided",
    "suppressed": "rate withheld: thin denominator",
    "unmeasurable": "no source",
}


def benchmark_state(entry):
    if not entry["measurable"]:
        return "unmeasurable"
    if entry["suppressed"]:
        return "suppressed"
    target, low, high = entry["target"], entry["wilson_low"], entry["wilson_high"]
    if target is None or low is None or high is None:
        return "undecided"
    if low > target:
        return "met"
    if high < target:
        return "not met"
    return "undecided"


def plot_benchmark_card(name, entry, series=None, platform=None, title=None):
    state = benchmark_state(entry)
    badge = BENCHMARK_BADGE[state]
    label = title or entry.get("label", name.replace("_", " ").title())
    decided = state in ("met", "not met", "undecided")
    has_spark = series is not None and not series.empty
    rows = 3 if has_spark else 1
    fig = make_subplots(
        rows=rows, cols=1,
        row_heights=[0.50, 0.32, 0.18] if has_spark else [1.0],
        vertical_spacing=0.18, shared_xaxes=False)

    if decided:
        fig.add_trace(go.Bar(
            x=[entry["value"]], y=[label], orientation="h", width=0.34,
            marker=dict(color=ink("series_1"), line=dict(width=0)),
            error_x=dict(
                type="data", symmetric=False,
                array=[entry["wilson_high"] - entry["value"]],
                arrayminus=[entry["value"] - entry["wilson_low"]],
                color=ink("text_secondary"), thickness=2, width=6),
            hovertemplate=(
                f"<b>{label}</b><br>value %{{x:.1%}}<br>"
                f"95% CI {pct(entry['wilson_low'], 1)}-{pct(entry['wilson_high'], 1)}<br>"
                f"{entry['numerator']}/{entry['denominator']} · {entry['participants']} participants"
                "<extra></extra>"),
            showlegend=False), row=1, col=1)
        headline = pct(entry["value"], 1)
    else:
        # A suppressed tile used to draw a fully transparent bar, leaving an
        # empty axis. During Phase 1 that is most of the board. Show the raw
        # counts as a dot on a count axis so the tile still moves.
        numerator = entry["numerator"] or 0
        denominator = entry["denominator"] or 0
        fig.add_trace(go.Scatter(
            x=[numerator], y=[label], mode="markers",
            marker=dict(size=14, color=ink("series_1"),
                        line=dict(color=ink("surface"), width=2)),
            hovertemplate=f"{numerator} of {denominator}<extra></extra>",
            showlegend=False), row=1, col=1)
        if denominator:
            fig.add_shape(type="line", x0=0, x1=denominator, y0=0, y1=0,
                          yref="y", line=dict(color=ink("grid"), width=6), row=1, col=1)
        headline = f"{entry['numerator'] if entry['numerator'] is not None else '-'}" \
                   f"/{entry['denominator'] if entry['denominator'] is not None else '-'}"

    if entry["target"] is not None and decided:
        fig.add_shape(
            type="line", x0=entry["target"], x1=entry["target"], y0=-0.45, y1=0.45,
            line=dict(color=ink("text"), width=2, dash="dot"), row=1, col=1)
        fig.add_annotation(
            x=entry["target"], y=0.55, yref="y", text=f"target {pct(entry['target'])}",
            showarrow=False, font=dict(size=10, color=ink("muted")), row=1, col=1)

    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=26, showarrow=False,
        text=(f"<b style='font-size:20px'>{headline}</b>  "
              f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> "
              f"<span style='color:{ink('muted')}'>{BENCHMARK_DETAIL[state]}</span>"),
        font=dict(family=FONT, size=12, color=ink("text")))

    if has_spark:
        traces = platform if platform else [(None, ink("series_1"), "")]
        for column, colour, trace_label in traces:
            column = column or "value"
            fig.add_trace(go.Scatter(
                x=series["date"], y=series[column], mode="lines+markers",
                # Step, never spline: each point is a day-long aggregate, and
                # smoothing invents values on days that never had them.
                line=dict(color=colour, width=2, shape="hv"),
                marker=dict(size=5, color=colour),
                connectgaps=False, name=trace_label or label,
                hovertemplate=f"{trace_label or label}<br>%{{x|%b %d}}: %{{y:.0%}}<extra></extra>",
                showlegend=bool(platform)), row=2, col=1)
        drawn = [column or "value" for column, _, _ in traces]
        if series[drawn].isna().all().all():
            fig.add_annotation(
                x=0.5, y=0.5, xref="x2 domain", yref="y2 domain",
                text="14-day rate suppressed (thin denominator)",
                showarrow=False, font=dict(size=10, color=ink("muted")), row=2, col=1)

        # The denominator, on its own axis rather than a second scale on the
        # rate axis. A dual axis would invent a relationship between them.
        if "den" in series.columns:
            fig.add_trace(go.Scatter(
                x=series["date"], y=series["den"], mode="lines", fill="tozeroy",
                line=dict(color=ink("muted"), width=1, shape="hv"),
                fillcolor=ink("band"),
                hovertemplate="%{x|%b %d}: %{y} in denominator<extra></extra>",
                showlegend=False), row=3, col=1)

        fig.update_yaxes(range=[0, 1], tickformat=".0%", nticks=3,
                         tickfont=dict(size=9, color=ink("muted")),
                         title=dict(text="rate", font=dict(size=9, color=ink("muted"))),
                         row=2, col=1)
        fig.update_xaxes(showticklabels=False, linecolor=ink("grid"), row=2, col=1)
        fig.update_yaxes(rangemode="tozero", nticks=2,
                         tickfont=dict(size=9, color=ink("muted")),
                         title=dict(text="denom.", font=dict(size=9, color=ink("muted"))),
                         row=3, col=1)
        fig.update_xaxes(showticklabels=True, tickformat="%b %d", nticks=4,
                         linecolor=ink("grid"),
                         title=dict(text="trailing 14 days (local date)",
                                    font=dict(size=10, color=ink("muted"))),
                         row=3, col=1)

    if decided:
        fig.update_xaxes(range=[0, 1], tickformat=".0%", row=1, col=1)
    else:
        span = max(int(entry["denominator"] or 0), 1)
        fig.update_xaxes(rangemode="tozero", tickformat="d",
                         dtick=max(1, span // 5),
                         title=dict(text=f"count ({entry['unit']})",
                                    font=dict(size=10, color=ink("muted"))),
                         row=1, col=1)
    fig.update_yaxes(showticklabels=True, tickfont=dict(color=ink("text"), size=12), row=1, col=1)
    base_layout(fig, 300 if has_spark else 175,
                f"{label} - % of {entry['unit']}",
                margin=dict(l=150, r=40, t=92, b=52 if has_spark else 56))
    fig.update_layout(
        bargap=0.5, showlegend=bool(platform),
        legend=dict(orientation="h", y=-0.30, x=0, font=dict(size=10, color=ink("text_secondary"))))
    return fig


def benchmark_spark(board, name):
    """Rate plus its denominator over the trailing 14 days, for one benchmark."""
    series = board["series_14d"]
    if name not in SPARK_COLUMN:
        return None
    rate, _numerator, denominator = SPARK_COLUMN[name]
    frame_ = series[["date", rate, denominator]].rename(
        columns={rate: "value", denominator: "den"})
    return frame_


def plot_benchmark_cards(board):
    figures = {}
    platforms = platform_series_14d(board["phase"])
    for name, entry in board["benchmarks"].items():
        spark = benchmark_spark(board, name)
        platform = None
        if name == "prompt_response":
            spark = platforms.rename(columns={"ios": "ios_rate", "android": "android_rate"})
            spark["den"] = platforms["ios_den"] + platforms["android_den"]
            platform = [("ios_rate", ink("series_1"), "iOS"),
                        ("android_rate", ink("series_2"), "Android")]
        figures[name] = plot_benchmark_card(name, entry, spark, platform)
    return figures

In [4]:
# STAGE 1A — CONSOLIDATED FOREST PLOT
# Five benchmarks against their targets in one glance, in the dot-and-whisker
# idiom a PI already reads in papers. Five separate cards each carried their own
# axis and 150px margin, which made comparison a scrolling exercise.
#
# The units differ (% of slots, of delivered prompts, of participant-days, of
# participants), so the unit rides in the ROW LABEL. Putting it in the figure
# title would invite reading five commensurable numbers off one axis.
def plot_benchmark_forest(board):
    entries = list(board["benchmarks"].items())
    fig = go.Figure()
    labels, ticks = [], []

    for name, entry in reversed(entries):
        state = benchmark_state(entry)
        badge = BENCHMARK_BADGE[state]
        label = entry.get("label", name.replace("_", " ").title())
        row = f"{label}<br><span style='font-size:9px;color:{ink('muted')}'>{entry['unit']}</span>"
        labels.append(row)
        ticks.append((row, entry, state, badge))

    for row, entry, state, badge in ticks:
        if state in ("met", "not met", "undecided"):
            fig.add_trace(go.Scatter(
                x=[entry["wilson_low"], entry["wilson_high"]], y=[row, row], mode="lines",
                line=dict(color=ink("text_secondary"), width=2),
                hoverinfo="skip", showlegend=False))
            fig.add_trace(go.Scatter(
                x=[entry["value"]], y=[row], mode="markers",
                marker=dict(size=13, color=STATUS[badge],
                            line=dict(color=ink("surface"), width=2)),
                hovertemplate=(f"{entry['numerator']}/{entry['denominator']}"
                               f" · {entry['participants']} participants<extra></extra>"),
                showlegend=False))
        if entry["target"] is not None:
            fig.add_trace(go.Scatter(
                x=[entry["target"]], y=[row], mode="markers",
                marker=dict(symbol="line-ns", size=13,
                            line=dict(color=ink("text"), width=2)),
                hovertemplate=f"target {pct(entry['target'])}<extra></extra>",
                showlegend=False))

    for row, entry, state, badge in ticks:
        value = pct(entry["value"], 1) if state in ("met", "not met", "undecided") else "-"
        counts = (f"{entry['numerator']}/{entry['denominator']}"
                  if entry["numerator"] is not None else "no source")
        fig.add_annotation(
            x=1.0, xref="paper", xanchor="left", xshift=10, y=row, showarrow=False,
            text=(f"<b>{value}</b>  <span style='color:{ink('muted')}'>{counts}</span>  "
                  f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]} {state}</span>"),
            font=dict(family=FONT, size=11, color=ink("text")), align="left")

    fig.update_xaxes(range=[0, 1], tickformat=".0%", showgrid=True, gridcolor=ink("grid"),
                     title=dict(text="value with Wilson 95% CI (% of each row's own unit)",
                                font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(type="category", categoryorder="array", categoryarray=labels,
                     tickfont=dict(size=11, color=ink("text")))
    base_layout(fig, 300, f"Cohort benchmarks - {board['phase']}",
                margin=dict(l=210, r=250, t=72, b=56),
                context={"phase": board["phase"], "n": board["n_participants"]})
    return fig


def plot_benchmark_sparklines(board):
    """The five trends as one small-multiple row, kept separate from the forest."""
    names = [name for name in board["benchmarks"] if name in SPARK_COLUMN]
    fig = make_subplots(rows=1, cols=len(names), shared_yaxes=True,
                        horizontal_spacing=0.05,
                        subplot_titles=[board["benchmarks"][n].get("label", n) for n in names])
    for index, name in enumerate(names, start=1):
        spark = benchmark_spark(board, name)
        fig.add_trace(go.Scatter(
            x=spark["date"], y=spark["value"], mode="lines",
            line=dict(color=ink("series_1"), width=2, shape="hv"),
            connectgaps=False, hovertemplate="%{x|%b %d}: %{y:.0%}<extra></extra>",
            showlegend=False), row=1, col=index)
        target = board["benchmarks"][name]["target"]
        if target is not None:
            fig.add_hline(y=target, line=dict(color=ink("muted"), width=1, dash="dot"),
                          row=1, col=index)
        if spark["value"].isna().all():
            fig.add_annotation(x=0.5, y=0.5, xref=f"x{index if index > 1 else ''} domain",
                               yref=f"y{index if index > 1 else ''} domain",
                               text="suppressed", showarrow=False,
                               font=dict(size=9, color=ink("muted")))
        fig.update_xaxes(tickformat="%b %d", nticks=3, tickfont=dict(size=9),
                         linecolor=ink("grid"), row=1, col=index)
    fig.update_yaxes(range=[0, 1], tickformat=".0%", col=1,
                     title=dict(text="rate", font=dict(size=10, color=ink("muted"))))
    base_layout(fig, 220, "Trailing 14 days - dotted rule is each benchmark's target",
                margin=dict(l=70, r=30, t=76, b=50),
                context={"phase": board["phase"], "n": board["n_participants"]})
    for annotation in fig.layout.annotations[:len(names)]:
        annotation.update(font=dict(size=10, color=ink("text")))
    return fig

In [5]:
# STAGE 1B — ENROLLMENT FUNNEL
def plot_funnel(board):
    rows = board["funnel"].copy()
    stages = rows[rows["stage"].ne("withdrew")]
    withdrew = rows[rows["stage"].eq("withdrew")]
    ramp = ink("ordinal")

    fig = go.Figure()
    for index, row in enumerate(stages.to_dict("records")):
        measurable = bool(row["measurable"])
        value = row["n"] if measurable else 0
        fig.add_trace(go.Bar(
            x=[value], y=[row["stage"]], orientation="h", width=0.62,
            marker=dict(
                color=ramp[min(index, len(ramp) - 1)] if measurable else "rgba(0,0,0,0)",
                line=dict(color=ink("axis") if not measurable else ink("surface"), width=2)),
            hovertemplate=(f"<b>{row['stage']}</b><br>n = {row['n']}<extra></extra>" if measurable
                           else f"<b>{row['stage']}</b><br>not measurable<extra></extra>"),
            showlegend=False))
        fig.add_annotation(
            x=value, y=row["stage"], xanchor="left", xshift=8, showarrow=False,
            text=(f"<b>{int(row['n'])}</b>" if measurable else
                  f"<span style='color:{STATUS['warning']}'>{STATUS_ICON['warning']}</span> not recorded"),
            font=dict(size=12, color=ink("text")))

    if len(withdrew):
        row = withdrew.iloc[0]
        fig.add_trace(go.Bar(
            x=[row["n"]], y=["withdrew"], orientation="h", width=0.42,
            marker=dict(color=STATUS["critical"], line=dict(color=ink("surface"), width=2)),
            hovertemplate=f"<b>withdrew</b><br>n = {row['n']}<extra></extra>", showlegend=False))
        fig.add_annotation(
            x=row["n"], y="withdrew", xanchor="left", xshift=8, showarrow=False,
            text=(f"<span style='color:{STATUS['critical']}'>{STATUS_ICON['critical']}</span> "
                  f"<b>{int(row['n'])}</b> dropout"),
            font=dict(size=12, color=ink("text")))

    order = list(stages["stage"])[::-1]
    if len(withdrew):
        order = ["withdrew"] + order
    fig.update_yaxes(categoryorder="array", categoryarray=order,
                     tickfont=dict(color=ink("text"), size=12))
    fig.update_xaxes(showgrid=True, gridcolor=ink("grid"), rangemode="tozero",
                     title=dict(text="participants (count)",
                                font=dict(size=11, color=ink("muted"))))
    base_layout(fig, 250, f"Enrollment funnel - {board['phase']} (n={board['n_participants']})",
                margin=dict(l=150, r=120, t=50, b=30))
    fig.update_layout(bargap=0.35)
    return fig

In [6]:
# STAGE 1C — MRT INTEGRITY STRIP
# One figure, four rows, ONE shared 0-100% axis. Four separate gauges each
# rescaled its own axis (the send-rate one to value * 1.1), so bar lengths in
# adjacent panels were not comparable - the thing a strip exists to allow.
INTEGRITY_ROWS = (
    ("eligibility_rate", "Eligibility rate", ELIGIBILITY_BAND),
    ("send_rate", "Send rate", None),
    ("cap_hit_rate", "Cap-hit rate", (0.0, CAP_HIT_ALARM)),
    ("outcome_capture", "Outcome capture", (OUTCOME_CAPTURE_ALARM, 1.0)),
)


def integrity_badge(entry):
    if entry.get("contradiction") or entry.get("alarm"):
        return "critical"
    if entry.get("value") is None or entry.get("wilson_low") is None:
        return "warning"
    return "good"


def plot_integrity_panel(board):
    integrity = board["integrity"]
    labels = [label for _key, label, _band in INTEGRITY_ROWS][::-1]
    fig = go.Figure()

    for key, label, band in INTEGRITY_ROWS:
        entry = integrity[key]
        if band:
            fig.add_shape(type="rect", x0=band[0], x1=band[1],
                          y0=labels.index(label) - 0.42, y1=labels.index(label) + 0.42,
                          fillcolor=ink("band"), line=dict(width=0), layer="below")
        value = entry.get("value")
        if value is not None:
            fig.add_trace(go.Bar(
                x=[min(value, 1.0)], y=[label], orientation="h", width=0.34,
                marker=dict(color=ink("series_1"), line=dict(width=0)),
                error_x=(dict(type="data", symmetric=False,
                              array=[entry["wilson_high"] - value],
                              arrayminus=[value - entry["wilson_low"]],
                              color=ink("text_secondary"), thickness=2, width=6)
                         if entry.get("wilson_low") is not None else None),
                hovertemplate=(f"<b>{label}</b><br>%{{x:.1%}}<br>"
                               f"{entry.get('numerator')}/{entry.get('denominator')}"
                               "<extra></extra>"),
                showlegend=False))
        target = entry.get("target")
        if isinstance(target, (int, float)):
            fig.add_trace(go.Scatter(
                x=[target], y=[label], mode="markers",
                marker=dict(symbol="line-ns", size=20, line=dict(color=ink("text"), width=2)),
                hovertemplate=f"target {target:g}<extra></extra>", showlegend=False))

    for key, label, _band in INTEGRITY_ROWS:
        entry = integrity[key]
        badge = integrity_badge(entry)
        note = ""
        if entry.get("contradiction"):
            note = f"  {STATUS_ICON['critical']} k &gt; n"
        if entry.get("value") is not None and entry["value"] > 1:
            note += f"  (off-scale: {pct(entry['value'], 0)})"
        fig.add_annotation(
            x=1.0, xref="paper", xanchor="left", xshift=10, y=label, showarrow=False,
            text=(f"<b>{pct(entry.get('value'), 1)}</b>  "
                  f"<span style='color:{ink('muted')}'>"
                  f"{entry.get('numerator')}/{entry.get('denominator')}</span>  "
                  f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span>{note}"),
            font=dict(family=FONT, size=11, color=ink("text")), align="left")

    fig.update_xaxes(range=[0, 1], tickformat=".0%", showgrid=True, gridcolor=ink("grid"),
                     title=dict(text="rate (% of each gauge's own denominator); "
                                     "shaded span is the acceptable range",
                                font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(type="category", categoryorder="array", categoryarray=labels,
                     tickfont=dict(size=11, color=ink("text")))
    base_layout(fig, 270, f"MRT integrity - {integrity['window']}",
                margin=dict(l=150, r=230, t=72, b=62),
                context={"phase": board["phase"], "n": board["n_participants"]})
    fig.update_layout(bargap=0.45)
    return fig


def plot_randomization_ecdf(integrity, draws):
    """ECDF against Uniform(0,1) with the KS acceptance envelope drawn.

    With few draws the staircase sits visibly off the diagonal even under a
    perfect RNG, so the printed p-value was the only thing saying otherwise.
    The band makes "inside" readable without reading the statistic.
    """
    audit = integrity["randomization_audit"]
    n = audit["draws"] or 0
    fig = go.Figure()

    if n:
        # Asymptotic two-sided critical value at alpha = 0.05.
        d_crit = 1.35810 / math.sqrt(n)
        grid = [index / 100 for index in range(101)]
        upper = [min(1.0, value + d_crit) for value in grid]
        lower = [max(0.0, value - d_crit) for value in grid]
        fig.add_trace(go.Scatter(x=grid + grid[::-1], y=upper + lower[::-1],
                                 fill="toself", fillcolor=ink("band"),
                                 line=dict(width=0), hoverinfo="skip",
                                 name=f"KS acceptance band (alpha=0.05, n={n})",
                                 showlegend=True))
    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1], mode="lines", name="Uniform(0,1)",
        line=dict(color=ink("muted"), width=2, dash="dot"),
        hovertemplate="Uniform(0,1)<br>%{x:.2f}<extra></extra>"))

    values = sorted(draws)
    if values:
        steps = [(index + 1) / len(values) for index in range(len(values))]
        fig.add_trace(go.Scatter(
            x=values, y=steps, mode="lines", name="Observed draws",
            line=dict(color=ink("series_1"), width=2, shape="hv"),
            hovertemplate="Observed<br>draw %{x:.3f}<br>ECDF %{y:.2f}<extra></extra>"))
    else:
        fig.add_annotation(x=0.5, y=0.5, text="no draws recorded", showarrow=False,
                           font=dict(size=11, color=ink("muted")))

    alarm = audit["ks_p_value"] is not None and audit["ks_p_value"] < audit["alarm_p"]
    thin = n < KS_MIN_DRAWS
    badge = "critical" if (alarm or audit["mismatches"]) else ("warning" if thin else "good")
    if audit["ks_statistic"] is None:
        summary = "KS: no draws recorded"
    elif thin:
        summary = (f"KS D = {audit['ks_statistic']:.3f}, p = {audit['ks_p_value']:.3f} - "
                   f"{n} draw(s), too few to test")
    else:
        summary = f"KS D = {audit['ks_statistic']:.3f}, p = {audit['ks_p_value']:.3f}"
    if audit["mismatches"]:
        summary += f" · {audit['mismatches']} mismatch(es)"
    fig.add_annotation(
        x=1.0, y=1.0, xref="paper", yref="paper", xanchor="right", yanchor="bottom",
        yshift=14, showarrow=False,
        text=f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> {summary}",
        font=dict(family=FONT, size=12, color=ink("text")))

    fig.update_xaxes(range=[0, 1], title=dict(text="randomization_draw (0-1)",
                                              font=dict(size=11, color=ink("muted"))))
    fig.update_yaxes(range=[0, 1], showgrid=True, gridcolor=ink("grid"),
                     title=dict(text="cumulative share of draws",
                                font=dict(size=11, color=ink("muted"))))
    base_layout(fig, 330, "Randomization uniformity", margin=dict(l=70, r=40, t=54, b=76))
    fig.update_layout(showlegend=True,
                      legend=dict(orientation="h", y=-0.30, x=0,
                                  font=dict(size=10, color=ink("text_secondary"))))
    return fig


def unexplained_decisions(phase="all", days=14):
    """Decision points the engine cleared on volatility but never drew for.

    observed_mssd above the threshold it recorded, threshold_source 'engine',
    and no randomization draw taken. The strongest available evidence of an
    engine bug, so it belongs beside the other protocol violations.
    """
    end = today_local()
    total = 0
    for user_id in cohort_users(phase):
        for offset in range(days):
            total += sum(1 for point in mssd_lane(user_id, end - timedelta(days=offset))
                         if point["unexplained"])
    return total


def plot_violation_counters(integrity, runin_sent, unexplained=0):
    cooldown = integrity["cooldown"]
    eligibility = integrity["eligibility_rate"]
    tiles = [
        ("Cooldown violations", cooldown["violations"], cooldown["alarm"],
         f"consecutive sends &lt; {cooldown['threshold_min']} min"),
        ("Run-in prompts sent", runin_sent, runin_sent > 0,
         "protocol: no prompts before day 7"),
        ("Unexplained decisions", unexplained, unexplained > 0,
         "over threshold, engine-sourced, no draw"),
        ("Reason disagreement", eligibility["reason_eligible_but_no_draw"]
         + eligibility["reason_ineligible_with_draw"],
         eligibility["alarm_reason_disagreement"], "trigger_reason vs randomization_draw"),
    ]
    fig = make_subplots(rows=1, cols=len(tiles), horizontal_spacing=0.05)
    for index, (label, value, alarm, caption) in enumerate(tiles, start=1):
        suffix = "" if index == 1 else str(index)
        xref, yref = f"x{suffix} domain", f"y{suffix} domain"
        badge = "critical" if alarm else "good"
        fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers",
                                 marker=dict(size=0.1, color="rgba(0,0,0,0)"),
                                 hoverinfo="skip", showlegend=False), row=1, col=index)
        fig.add_annotation(
            x=0.5, y=0.88, xref=xref, yref=yref, showarrow=False, yanchor="top",
            text=f"<b style='font-size:30px;color:{STATUS[badge]}'>{int(value)}</b>",
            font=dict(family=FONT))
        fig.add_annotation(
            x=0.5, y=0.30, xref=xref, yref=yref, showarrow=False,
            text=(f"<span style='color:{STATUS[badge]}'>{STATUS_ICON[badge]}</span> "
                  f"<b>{label}</b>"),
            font=dict(family=FONT, size=12, color=ink("text")))
        fig.add_annotation(
            x=0.5, y=0.02, xref=xref, yref=yref, showarrow=False,
            text=caption, font=dict(family=FONT, size=10, color=ink("muted")))
        fig.update_xaxes(visible=False, row=1, col=index)
        fig.update_yaxes(visible=False, row=1, col=index)
    base_layout(fig, 200, "Protocol violations (counts; any non-zero is a breach)",
                margin=dict(l=20, r=20, t=54, b=18))
    return fig


def plot_integrity_strip(board):
    integrity = board["integrity"]
    rows = decisions(board["phase"])
    draws = (rows.loc[rows["randomization_draw"].notna(), "randomization_draw"]
             .astype(float).tolist() if not rows.empty else [])
    runin_sent = int(daily_grid_metrics(board["phase"]).query("is_run_in")["sent_n"].sum())
    return {
        "panel": plot_integrity_panel(board),
        "randomization": plot_randomization_ecdf(integrity, draws),
        "violations": plot_violation_counters(
            integrity, runin_sent, unexplained_decisions(board["phase"])),
    }

In [7]:
# STAGE 1D — ALERT LOG
SEVERITY_STATUS = {"critical": "critical", "high": "serious", "warning": "warning"}


def style_alert_log(feed=None, include_resolved=False):
    feed = alert_feed(include_resolved) if feed is None else feed
    if feed.empty:
        return pd.DataFrame(columns=["severity", "rule_id", "scope", "fired_at"]).style
    table = feed.copy()
    table["severity"] = [
        f"{STATUS_ICON[SEVERITY_STATUS[s]]} {s}" for s in table["severity"]]
    table["scope"] = [
        f"{'COHORT' if scope == 'cohort' else 'PARTICIPANT'}" for scope in table["scope"]]
    table["participant"] = [
        "-" if pd.isna(uid) else str(int(uid)) for uid in table["user_id"]]
    table["fired_at"] = pd.to_datetime(table["fired_at"], utc=True).dt.tz_convert(
        PARTICIPANT_TZ).dt.strftime("%b %d %H:%M")
    table["detail"] = [json.dumps(payload or {})[:80] for payload in table["payload"]]
    columns = ["severity", "rule_id", "scope", "participant", "fired_at", "link_date", "detail"]

    def tag(value):
        for name, role in SEVERITY_STATUS.items():
            if value.endswith(name):
                return (f"color:{STATUS[role]};font-weight:600;"
                        f"font-family:{FONT};white-space:nowrap")
        return ""

    def scope_tag(value):
        colour = ink("muted") if value == "COHORT" else ink("text_secondary")
        return f"color:{colour};font-size:11px;letter-spacing:.04em"

    return (
        table[columns].style
        .map(tag, subset=["severity"])
        .map(scope_tag, subset=["scope"])
        .set_properties(**{"font-family": FONT, "font-size": "12px",
                           "color": ink("text_secondary"), "background-color": ink("surface")})
        .set_table_styles([
            {"selector": "th", "props": [("font-family", FONT), ("font-size", "11px"),
                                         ("color", ink("muted")), ("text-align", "left"),
                                         ("border-bottom", f"1px solid {ink('axis')}")]},
            {"selector": "td", "props": [("border-bottom", f"1px solid {ink('grid')}")]},
        ])
        .hide(axis="index")
    )

## 1A - Benchmark tiles

In [8]:
# Five benchmarks, one axis, dot-and-whisker with a target tick per row - the
# idiom a PI reads in papers. Colour now follows the INTERVAL, not the point
# estimate: green only when the CI clears the target, red only when it falls
# entirely below, amber when it spans the target and the question is still open.
# Units differ between rows, so each row label carries its own; the axis is
# "% of that row's unit", never five commensurable numbers.
# Today: wear is met (CI 0.95-0.99 above 0.80), prompt response is met
# (0.64-0.86 above 0.70), slot coverage is not met (0.46-0.51, entirely below
# 0.75), and retention and hair sample are undecided for want of a denominator.
plot_benchmark_forest(board)

In [9]:
# The same five as trends, kept as a separate small-multiple row rather than
# crammed under each tile. Step lines, never splines: each point is a day-long
# aggregate and smoothing would invent values on days that never had them.
# Dotted rule on each panel is that benchmark's target.
plot_benchmark_sparklines(board)

In [10]:
# The individual card still exists when you want one benchmark in full: bullet
# with its CI, the 14-day rate, and - new - the DENOMINATOR drawn beneath it.
# During the Phase 1 ramp the cohort grows from 5 to 40, so a rising rate line
# and a rising denominator look identical unless both are on screen. Here the
# denominator is flat at 78 slots/day, so the rate really is the rate.
cards = plot_benchmark_cards(board)
cards["slot_coverage"]

In [11]:
# Formal retention needs day 35 and only one participant has reached it, so the
# rate is suppressed. A suppressed tile now plots its raw counts as a dot on a
# COUNT axis rather than drawing an invisible bar on a percent axis - during
# Phase 1 that is most of the board, and an empty axis tells the reader nothing.
cards["retention"]

In [12]:
# Unmeasurable, not zero, and now visibly so: no source exists, so there is no
# dot to place and the badge says why. Nothing in the schema records hair-sample
# collection; it needs an RA sheet loaded into a derived table first.
cards["hair_sample"]

## 1B - Enrolment funnel

In [13]:
# Horizontal stages, withdrawals as a separate offshoot rather than a stage.
# 14 started day 1, 13 active today, 1 completed day 34, 1 withdrew.
# 'consented' is drawn as an empty outlined bar labelled 'not recorded': User
# has enrolled_at and is_enrolled and nothing before them, so reporting
# enrolment as consent would be a fabricated number.
plot_funnel(board)

## 1C - MRT integrity strip

This strip, not the benchmark tiles, is what says whether the trial will be
analysable.

In [14]:
# All four gauges on ONE shared 0-100% axis, so bar lengths are comparable
# between rows. They were four separate figures, and the send-rate one rescaled
# its axis to value * 1.1, which made adjacent bars silently incomparable.
# Shaded spans are each gauge's acceptable range: 10-35% eligibility, cap-hit
# under 10%, outcome capture above 60%. The tick is the target where one exists.
strip = plot_integrity_strip(board)
strip["panel"]

In [15]:
# The ECDF of every randomization draw against Uniform(0,1), now with the KS
# acceptance envelope at alpha = 0.05 drawn as a shaded band. With few draws a
# staircase sits visibly off the diagonal even under a perfect RNG, so the
# p-value used to be the only thing saying the randomiser was fine. Inside the
# band is now readable without reading the statistic.
strip["randomization"]

In [16]:
# Counters, not rates: any non-zero value is a protocol breach. The fourth tile
# is new - "unexplained decisions" counts points where observed MSSD cleared the
# threshold the ENGINE itself recorded, yet no randomization draw was taken.
# That is the strongest evidence of an engine bug the data can offer, and it was
# computed in mssd_lane but never surfaced outside the Stage 3 timeline.
strip["violations"]

## 1D - Alert feed

In [17]:
# Newest first within severity. A null participant means a COHORT-scoped alert:
# runin_violation and sync_stale are facts about the engine and the pipeline,
# and raising them per participant would put an identical critical on everyone
# and flatten the risk score.
# 7 open: 2 cohort-scoped criticals, plus cooldown_violation on 1007,
# cap_exceeded on 1008, no_ema_48h on 1009, wear_low on 1010 and
# slot_coverage_low on 1011 - one participant per rule.
style_alert_log()

severity,rule_id,scope,participant,fired_at,link_date,detail
■ critical,runin_violation,COHORT,-,Sep 16 06:00,2026-09-15,"{""detail"": ""engine has no run-in gate"", ""date"": ""2026-09-15""}"
■ critical,sync_stale,COHORT,-,Sep 16 06:00,2026-09-15,"{""measurable"": false, ""date"": ""2026-09-15""}"
■ critical,cooldown_violation,PARTICIPANT,1007,Sep 16 06:00,2026-09-15,"{""gap_min"": 28, ""date"": ""2026-09-15""}"
■ critical,cap_exceeded,PARTICIPANT,1008,Sep 16 06:00,2026-09-15,"{""delivered_n"": 6, ""date"": ""2026-09-15""}"
▲ high,no_ema_48h,PARTICIPANT,1009,Sep 16 06:00,2026-09-15,"{""hours"": 62, ""date"": ""2026-09-15""}"
▲ high,wear_low,PARTICIPANT,1010,Sep 16 06:00,2026-09-15,"{""consecutive_days"": 3, ""date"": ""2026-09-15""}"
▲ warning,slot_coverage_low,PARTICIPANT,1011,Sep 16 06:00,2026-09-15,"{""coverage"": 0.31, ""date"": ""2026-09-15""}"


In [18]:
# The same five tiles as a table, for the numbers behind the bullets.
benchmark_table(board)

,tile,measurable,suppressed,value,target,wilson_low,wilson_high,numerator,denominator,participants,unit
0,Slot coverage (proxy),True,False,0.488932,0.75,0.463992,0.513927,751.0,1536.0,14,check-in slots covered
1,prompt_response,True,False,0.769231,0.70,0.638662,0.862757,40.0,52.0,10,delivered prompts responded to
2,wear,True,False,0.981928,0.80,0.948220,0.993835,163.0,166.0,14,participant-days meeting the coverage target
3,retention,True,True,NaN,0.85,NaN,NaN,1.0,1.0,1,participants retained at day 35
4,Hair sample,False,False,NaN,0.90,NaN,NaN,NaN,NaN,0,hair samples collected


In [19]:
# Every rule in the alert spec evaluated against the current data, with the
# severity ladder the spec defines (high / medium / low / info).
findings = evaluate_rules("all")
print("findings:", len(findings))
findings.groupby(["severity", "rule_id"]).size()

findings: 60


severity  rule_id            
high      cap_exceeded            2
          cooldown_violation      9
          runin_prompt_sent      12
          scheduler_silent       21
info      enrollment_flip         1
low       clock_skew              2
medium    mssd_signal_missing     1
          no_checkin_72h          2
          sync_stale              9
          wear_low                1
dtype: int64